<style>
@page {
    size: A5;
    margin: 18mm 16mm 18mm 16mm;
}

body {
    font-family: "DejaVu Sans", "Arial", sans-serif;
    font-size: 12px;
    line-height: 1.35;
}

.jp-Notebook {
    max-width: 180mm;
    margin: 0 auto;
}

.jp-Cell {
    page-break-inside: avoid;
    break-inside: avoid;
}

pre, code {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}

table {
    width: 100%;
    table-layout: fixed;
    word-wrap: break-word;
}

img, svg, canvas {
    max-width: 100%;
    height: auto;
}

.output_area pre {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}
</style>

In [55]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave
from scipy.optimize import minimize

np.random.seed(777)

In [56]:
n = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

alpha = 0.05
precision = 1001

# Функция распределения Колмогорова
def kolmogorov(x):
    return 1 + np.sum([2 * (-1)**k * np.exp(-2 * (k*k) * (x*x)) for k in range(1, precision)])

### a) Критерий Колмогорова

In [57]:
f_emp = np.cumsum(m) / n
f_th = np.arange(10) / 10

f_emp_prev = np.concatenate(([0], f_emp[:-1]))

d_plus = np.abs(f_emp - f_th)
d_minus = np.abs(f_emp_prev - f_th)
d_n = np.max(np.maximum(d_plus, d_minus))

delta_tilde = d_n * np.sqrt(n)
p_val = 1 - kolmogorov(delta_tilde)

print(f"Статистика критерия Колмогорова (Δ с волной) = {delta_tilde:.2f}, p-value = {p_val:.4f}")
if (p_val > alpha):
    print(f"{p_val:.4f} > {alpha} => Нет веских причин отвергнуть H_0.")
else:
    print(f"{p_val:.4f} <= {alpha} => Отвергаем H_0.")

Статистика критерия Колмогорова (Δ с волной) = 1.40, p-value = 0.0397
0.0397 <= 0.05 => Отвергаем H_0.


В первом случае, с критерием $ \chi^2 $ не было веских причин отвергнуть гипотезу $ H_0 $. Теперь же, с критерием Колмогорова, мы получили более точный результат, а именно смогли **опровергнуть** гипотезу.

### б) Критерий Колмогорова

In [60]:
def get_pk(theta1, theta2):
    edges = np.arange(10) - 0.5
    cdf_values = st.norm.cdf(edges[1:], loc=theta1, scale=theta2)
    
    pk = np.zeros(10)
    pk[0] = cdf_values[0]
    pk[1:9] = np.diff(cdf_values)
    pk[9] = 1 - cdf_values[-1]
    return pk

def neg_log_likelihood(params):
    t1, t2 = params
    if t2 <= 0: return 1e10
    pk = get_pk(t1, t2)
    return -np.sum(m * np.log(pk + 1e-12))

theta1_start = 1/n * np.sum(np.arange(10) * m)
theta2_start = np.sqrt(1/n * np.sum(np.arange(10)**2 * m) - theta1_start**2)

res = minimize(neg_log_likelihood, x0=[theta1_start, theta2_start], method='Nelder-Mead')

theta1_ompg, theta2_ompg = res.x
pk_ompg = get_pk(theta1_ompg, theta2_ompg)
delta_ompg = np.sum((m - n * pk_ompg)**2 / (n * pk_ompg))

p_val_ompg = 1 - st.chi2.cdf(delta_ompg, 7)

print(f"ОМПГ параметры: theta1 = {theta1_ompg:.3f}, theta2 = {theta2_ompg:.3f}")
print(f"Статистика хи-квадрат (ОМПГ) = {delta_ompg:.3f}")
print(f"p-value = {p_val_ompg:.4f}")

if p_val_ompg > alpha:
    print(f"{p_val_ompg:.4f} > {alpha} => Нет веских причин отвергнуть H_0.")
else:
    print(f"{p_val_ompg:.4f} <= {alpha} => Отвергаем H_0.")

ОМПГ параметры: theta1 = 4.790, theta2 = 2.680
Статистика хи-квадрат (ОМПГ) = 9.803
p-value = 0.2000
0.2000 > 0.05 => Нет веских причин отвергнуть H_0.


In [59]:
n_bootstrap = 50000

f_emp = np.cumsum(m) / n
f_emp_prev = np.concatenate(([0], f_emp[:-1]))

f_th = st.norm.cdf(np.arange(10), loc=theta1_ompg, scale=theta2_ompg)

d_plus = np.abs(f_emp - f_th)
d_minus = np.abs(f_emp_prev - f_th)
d_n_data = np.max(np.maximum(d_plus, d_minus))
delta_data = d_n_data * np.sqrt(n)

def compute_bs_deltas(n_samples, n_size, mu, sigma):
    samples = np.random.normal(loc=mu, scale=sigma, size=(n_samples, n_size))
    samples.sort(axis=1)
    
    bs_means = np.mean(samples, axis=1, keepdims=True)
    bs_stds = np.std(samples, axis=1, ddof=1, keepdims=True)
    
    f_emp_minus = np.arange(0, n_size) / n_size
    f_emp_plus = np.arange(1, n_size + 1) / n_size
    
    f_th_bs = st.norm.cdf(samples, loc=bs_means, scale=bs_stds)
    
    d_plus_bs = np.abs(f_emp_plus - f_th_bs)
    d_minus_bs = np.abs(f_emp_minus - f_th_bs)

    d_n_bs = np.max(np.maximum(d_plus_bs, d_minus_bs), axis=1)
    
    return d_n_bs * np.sqrt(n_size)

batch_size = 10000
bs_deltas = np.concatenate([
    compute_bs_deltas(batch_size, n, theta1_ompg, theta2_ompg) 
    for _ in range(n_bootstrap // batch_size)
])

pval_kolmogorov_b = np.sum(bs_deltas >= delta_data) / n_bootstrap

print(f"Статистика критерия (Δ с волной) = {delta_data:.4f}")
print(f"p-value = {pval_kolmogorov_b:.4f}")

if pval_kolmogorov_b > alpha:
    print(f"{pval_kolmogorov_b:.4f} > {alpha} => Нет веских причин отвергнуть H_0.")
else:
    print(f"{pval_kolmogorov_b:.4f} <= {alpha} => Отвергаем H_0.")

Статистика критерия (Δ с волной) = 0.9871
p-value = 0.0177
0.0177 <= 0.05 => Отвергаем H_0.


Аналогично. С критерием $ \chi^2 $ не было веских причин отвергнуть гипотезу $ H_0 $. Теперь же смогли **опровергнуть** гипотезу.